# 01a — Replogle 2022: QC and Preprocessing

**Dataset:** ReplogleWeissman2022_rpe1 (scPerturb-harmonized)  
**Accession:** GSE181897  
**Phase:** 2  
**Date:** 2026-03-11  
**Objective:** Apply cell/gene QC filters, assess cells-per-perturbation distribution, flag ambiguous guide assignments, and produce a clean AnnData for downstream analysis.

## Table of Contents

1. [Setup](#setup)
2. [Data Loading](#data-loading) — download from S3 / scPerturb, initial inspection
3. [Cell QC](#cell-qc) — UMI count, genes detected, mitochondrial fraction
   - 3a. Compute and visualise QC metrics
   - 3b. Apply thresholds and report removed cells
4. [Gene QC](#gene-qc) — filter lowly detected genes
5. [Cells-per-Perturbation Distribution](#cpp) — statistical power, fitness effects
6. [Guide Assignment QC](#guide-qc) — nperts, coverage, ambiguity flags
7. [Control Fraction Check](#ctrl-frac)
8. [Post-QC Preprocessing](#preproc) — normalise, HVG, scale, PCA
9. [Save Processed AnnData](#save)
10. [Summary & Phase 2 checkpoint](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
import boto3
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)

from pathlib import Path
DATA_DIR   = Path('../../data/raw/scperturb')
PROC_DIR   = Path('../../data/processed/scperturb')
RESULTS    = Path('../../results')
FIG_DIR    = RESULTS / 'figures'
TBL_DIR    = RESULTS / 'tables'
for d in [DATA_DIR, PROC_DIR, FIG_DIR, TBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET  = 'learn-perturb-seq'
H5AD_FILE  = 'ReplogleWeissman2022_rpe1.h5ad'
LOCAL_PATH = DATA_DIR / H5AD_FILE
PROC_PATH  = PROC_DIR / 'ReplogleWeissman2022_rpe1_qc.h5ad'

# Project-wide QC thresholds (from CLAUDE.md)
MIN_UMIS_PER_CELL   = 1_000
MIN_GENES_PER_CELL  = 500
MAX_PCT_MITO        = 20.0
MIN_CELLS_PER_PERT  = 20    # hard minimum: exclude perturbations below this
CELL_PER_PERT_WARN  = 200   # soft warning: E-distance unreliable below this
MIN_GENE_CELLS      = 10    # minimum cells a gene must be detected in
CONTROL_FRAC_MIN    = 0.10
CONTROL_FRAC_MAX    = 0.30
N_HVG               = 2_000
N_PCA               = 50

print('Setup complete.')
print(f'QC thresholds:')
print(f'  MIN_UMIS_PER_CELL  = {MIN_UMIS_PER_CELL:,}')
print(f'  MIN_GENES_PER_CELL = {MIN_GENES_PER_CELL:,}')
print(f'  MAX_PCT_MITO       = {MAX_PCT_MITO}%')
print(f'  MIN_CELLS_PER_PERT = {MIN_CELLS_PER_PERT}')
print(f'  MIN_GENE_CELLS     = {MIN_GENE_CELLS}')

<a id='data-loading'></a>
## 1. Data Loading

### About the dataset

**ReplogleWeissman2022_rpe1** is a genome-scale CRISPRi screen in RPE1 cells published in:

> Replogle, J.M. et al. (2022). Mapping information-rich genotype-phenotype landscapes with
> genome-scale Perturb-seq. *Cell* 185, 2559–2575.

**Key facts:**
- **Cell type:** RPE1 (retinal pigment epithelium) — non-cancerous, near-diploid, p53 wild-type
- **Perturbation type:** CRISPRi (dCas9-KRAB repression of transcription start sites)
- **Scale:** ~200K cells covering ~5,000–10,000 gene knockdowns
- **Why RPE1?** Near-diploid genome, low baseline expression noise, well-validated CRISPRi efficiency

The scPerturb-harmonized version standardizes all field names to the conventions used across the portal.

In [ ]:
if LOCAL_PATH.exists():
    print(f'Using cached file: {LOCAL_PATH} ({LOCAL_PATH.stat().st_size / 1e9:.1f} GB)')
else:
    # Try S3 first
    s3 = boto3.client('s3')
    try:
        head = s3.head_object(Bucket=S3_BUCKET, Key=f'raw/scperturb/{H5AD_FILE}')
        size_gb = head['ContentLength'] / 1e9
        print(f'Downloading from S3 ({size_gb:.1f} GB) ...')
        s3.download_file(S3_BUCKET, f'raw/scperturb/{H5AD_FILE}', str(LOCAL_PATH))
        print('Download complete.')
    except Exception as e:
        print(f'S3 download failed: {e}')
        print()
        print('MANUAL DOWNLOAD INSTRUCTIONS:')
        print('  1. Go to: https://zenodo.org/record/7041849')
        print('     (scPerturb compendium — Zenodo record 7041849)')
        print(f'  2. Download: {H5AD_FILE}')
        print(f'  3. Move to:  {LOCAL_PATH}')
        print(f'  4. Upload to S3:')
        print(f'     aws s3 cp {LOCAL_PATH} s3://{S3_BUCKET}/raw/scperturb/{H5AD_FILE}')
        raise FileNotFoundError(f'{H5AD_FILE} not available. See instructions above.')

adata = sc.read_h5ad(LOCAL_PATH)
print(adata)

In [ ]:
print('=== Initial dataset summary ===')
print(f'Cells x Genes: {adata.shape}')
print(f'obs columns:   {adata.obs.columns.tolist()}')
print(f'var columns:   {adata.var.columns.tolist()}')
print()
print('=== scPerturb standardized fields ===')
for field in ['perturbation', 'perturbation_type', 'nperts', 'cell_line', 'cancer']:
    if field in adata.obs.columns:
        vals = adata.obs[field].unique()
        print(f'  {field:22s}: {vals[:5].tolist()}{"..." if len(vals) > 5 else ""}')
    else:
        print(f'  {field:22s}: MISSING')
print()
print('=== Key count fields ===')
for field in ['ncounts', 'ngenes', 'percent_mito', 'percent_ribo',
              'coverage', 'good_coverage', 'guide_id']:
    if field in adata.obs.columns:
        col = adata.obs[field]
        if col.dtype in [float, int, 'float32', 'float64', 'int32', 'int64']:
            print(f'  {field:20s}: min={col.min():.1f}, median={col.median():.1f}, max={col.max():.1f}')
        else:
            print(f'  {field:20s}: dtype={col.dtype}, nunique={col.nunique()}')
    else:
        print(f'  {field:20s}: NOT IN OBS')
print()
print('=== Perturbation overview ===')
n_perts = adata.obs['perturbation'].nunique()
n_ctrl  = (adata.obs['perturbation'] == 'control').sum()
print(f'  Total cells:                {len(adata):>10,}')
print(f'  Unique perturbations:       {n_perts:>10,}')
print(f'  Control cells:              {n_ctrl:>10,}  ({n_ctrl/len(adata):.1%})')

<a id='cell-qc'></a>
## 2. Cell QC — filtering low-quality cells

Cell QC removes four classes of problematic observations from the count matrix before
any biological analysis. Each metric captures a distinct failure mode:

---

### Metric 1: Total UMI count per cell (`ncounts` / `total_counts`)

The total number of unique molecular identifiers (UMIs) detected in a cell reflects
how much mRNA was captured and sequenced. The distribution is approximately log-normal
in a healthy experiment.

**Too few UMIs → exclude:**
- **Empty droplets:** In 10x Chromium, some gel-bead-in-emulsion (GEM) droplets contain no
  cells — just ambient RNA from the cell suspension. These register a small number of UMIs
  (typically < 500) from free-floating mRNA in the media. Knee-point calling (CellRanger's
  algorithm) usually removes them, but stragglers can remain.
- **Damaged / dying cells:** Cells with ruptured membranes lose cytoplasmic mRNA before
  capture, yielding low UMI counts.
- **Failed library preparation:** Cells that were encapsulated but whose library prep failed.

**Too many UMIs → suspect doublet:**
- **Doublets:** Two cells captured in the same GEM droplet. The library represents the
  combined transcriptome of two cells, so total UMIs ≈ 2× the expected single-cell count.
- Typical doublet rate: 1–8% per 1,000 cells loaded. At genome scale (Replogle 2022 loaded
  ~100K cells), doublets represent a meaningful fraction.
- We apply a data-driven upper cutoff here (median + 3×MAD on log-scale) as a first-pass
  filter. Formal doublet detection (scrublet) is deferred to Phase 4.

**Threshold used:** `MIN_UMIS_PER_CELL = 1,000` (lower bound); dynamic upper bound.

---

### Metric 2: Genes detected per cell (`ngenes` / `n_genes_by_counts`)

The number of distinct genes with at least 1 UMI. Highly correlated with total UMI count,
but captures additional information:
- Cells with the same UMI count but very few genes may have their counts concentrated in
  a single highly expressed gene (e.g., a haemoglobin gene in an erythroid contamination).
- Very few genes detected: same failure modes as low UMI count.
- Very many genes detected: consistent with a doublet (combines two cells' gene repertoires).

**Threshold used:** `MIN_GENES_PER_CELL = 500`

---

### Metric 3: Mitochondrial fraction (`percent_mito`)

The fraction of total UMIs derived from mitochondria-encoded genes (gene names starting
with `MT-` in human). Mitochondria have their own genome and produce 13 essential
respiratory chain proteins plus ribosomal RNA.

**Why high MT% indicates low quality:**
1. When a cell's plasma membrane is damaged (stress, apoptosis, or mechanical disruption
   during dissociation), **cytoplasmic mRNA leaks out** of the cell.
2. Mitochondria are membrane-enclosed organelles — their RNA is retained even after the
   cell membrane ruptures.
3. Result: the captured library becomes depleted of cytoplasmic (protein-coding) transcripts
   but retains MT transcripts, inflating the MT fraction.

**Cell-type considerations:**
- RPE1 cells (retinal pigment epithelium) are metabolically active but not exceptionally
  mitochondria-rich. The 20% threshold is appropriate.
- For cardiomyocytes or neurons (high mitochondrial mass), a higher threshold (30–40%) may
  be needed — this is a dataset-specific decision.

**Threshold used:** `MAX_PCT_MITO = 20%`

---

### Applying multiple filters simultaneously

All three filters are applied with AND logic — a cell must pass **all** thresholds to be
retained. This means some cells might be near-boundary on one metric; visualising the joint
distribution (scatter plots, coloured by MT%) is essential for choosing appropriate cutoffs.

In [ ]:
# ---- Compute or verify QC metrics ----------------------------------------
# scPerturb datasets pre-compute ncounts, ngenes, percent_mito.
# If missing, compute them now with scanpy.
if 'ncounts' not in adata.obs.columns and 'total_counts' not in adata.obs.columns:
    print('Computing QC metrics...')
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
    )
    adata.obs['ncounts']     = adata.obs['total_counts']
    adata.obs['ngenes']      = adata.obs['n_genes_by_counts']
    adata.obs['percent_mito'] = adata.obs['pct_counts_mt']
else:
    print('Using pre-computed QC metrics from scPerturb harmonization.')

# Use consistent column names
col_umi  = 'ncounts'     if 'ncounts'     in adata.obs.columns else 'total_counts'
col_gene = 'ngenes'      if 'ngenes'      in adata.obs.columns else 'n_genes_by_counts'
col_mito = 'percent_mito'

# Summary statistics before filtering
print(f'\nQC metric summary (BEFORE filtering, n={len(adata):,} cells):')
print(f'  {col_umi:15s}: median={adata.obs[col_umi].median():,.0f}, '
      f'mean={adata.obs[col_umi].mean():,.0f}, '
      f'range=[{adata.obs[col_umi].min():,.0f}, {adata.obs[col_umi].max():,.0f}]')
print(f'  {col_gene:15s}: median={adata.obs[col_gene].median():.0f}, '
      f'mean={adata.obs[col_gene].mean():.0f}, '
      f'range=[{adata.obs[col_gene].min():.0f}, {adata.obs[col_gene].max():.0f}]')
print(f'  {col_mito:15s}: median={adata.obs[col_mito].median():.1f}%, '
      f'mean={adata.obs[col_mito].mean():.1f}%, '
      f'range=[{adata.obs[col_mito].min():.1f}%, {adata.obs[col_mito].max():.1f}%]')

# Dynamic upper UMI cutoff: median + 3*MAD on log-scale
# MAD (median absolute deviation) is robust to outliers; 3*MAD captures ~99%
# of a normal distribution without being fooled by the long upper tail.
log_umi  = np.log1p(adata.obs[col_umi])
mad_umi  = np.median(np.abs(log_umi - log_umi.median()))
upper_umi_log = log_umi.median() + 3 * mad_umi
upper_umi = int(np.expm1(upper_umi_log))
print(f'\nDynamic upper UMI cutoff (median + 3*MAD on log scale): {upper_umi:,}')

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ---- Row 1: Histograms of each QC metric ----------------------------------
ax0 = fig.add_subplot(gs[0, 0])
ax0.hist(np.log1p(adata.obs[col_umi]), bins=100, color='steelblue', edgecolor='none')
ax0.axvline(np.log1p(MIN_UMIS_PER_CELL), color='red',  ls='--', lw=1.5, label=f'min={MIN_UMIS_PER_CELL:,}')
ax0.axvline(np.log1p(upper_umi),         color='orange',ls='--', lw=1.5, label=f'max={upper_umi:,}')
ax0.set_xlabel('log1p(total UMI count)')
ax0.set_ylabel('Cells')
ax0.set_title('Total UMI count distribution')
ax0.legend(fontsize=8)

ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(adata.obs[col_gene], bins=100, color='steelblue', edgecolor='none')
ax1.axvline(MIN_GENES_PER_CELL, color='red', ls='--', lw=1.5, label=f'min={MIN_GENES_PER_CELL}')
ax1.set_xlabel('Genes detected')
ax1.set_ylabel('Cells')
ax1.set_title('Genes detected per cell')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(adata.obs[col_mito], bins=100, color='steelblue', edgecolor='none')
ax2.axvline(MAX_PCT_MITO, color='red', ls='--', lw=1.5, label=f'max={MAX_PCT_MITO}%')
ax2.set_xlabel('Mitochondrial fraction (%)')
ax2.set_ylabel('Cells')
ax2.set_title('Mitochondrial content')
ax2.legend(fontsize=8)

# ---- Row 2: Joint scatter (UMI vs genes, coloured by MT%) -----------------
# Joint plots are essential for catching correlations between QC metrics.
# A normal cell cloud forms a compact, positively-correlated arc (UMI vs genes).
# Cells far below the main cloud = low quality.
# Cells far above = potential doublets.
# Cells with normal UMI/gene counts but high MT% = stressed/dying cells.
sample_n = min(50_000, len(adata))   # subsample for scatter plot speed
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(adata), size=sample_n, replace=False)
umi_s  = adata.obs[col_umi].values[sample_idx]
gene_s = adata.obs[col_gene].values[sample_idx]
mito_s = adata.obs[col_mito].values[sample_idx]

ax3 = fig.add_subplot(gs[1, 0:2])
sc_plot = ax3.scatter(
    np.log1p(umi_s), gene_s,
    c=mito_s, cmap='RdYlGn_r', s=3, alpha=0.4,
    vmin=0, vmax=30, rasterized=True
)
plt.colorbar(sc_plot, ax=ax3, label='MT fraction (%)')
ax3.axvline(np.log1p(MIN_UMIS_PER_CELL), color='red',   ls='--', lw=1, alpha=0.7)
ax3.axvline(np.log1p(upper_umi),         color='orange', ls='--', lw=1, alpha=0.7)
ax3.axhline(MIN_GENES_PER_CELL,          color='blue',   ls='--', lw=1, alpha=0.7,
            label=f'gene min={MIN_GENES_PER_CELL}')
ax3.set_xlabel('log1p(total UMI count)')
ax3.set_ylabel('Genes detected')
ax3.set_title(f'Joint QC plot (subsample n={sample_n:,}, coloured by MT%)')
ax3.legend(fontsize=8)

# MT% vs UMI (to see high-MT cells within the UMI range)
ax4 = fig.add_subplot(gs[1, 2])
ax4.scatter(
    np.log1p(umi_s), mito_s,
    c=(mito_s > MAX_PCT_MITO).astype(int),
    cmap='RdYlGn_r', s=3, alpha=0.3, rasterized=True
)
ax4.axhline(MAX_PCT_MITO, color='red', ls='--', lw=1.5, label=f'MT max={MAX_PCT_MITO}%')
ax4.set_xlabel('log1p(total UMI count)')
ax4.set_ylabel('Mitochondrial fraction (%)')
ax4.set_title('MT% vs UMI count')
ax4.legend(fontsize=8)

plt.suptitle('Cell QC metrics — ReplogleWeissman2022_rpe1 (pre-filter)', y=1.01, fontsize=13)
plt.savefig(FIG_DIR / '01a_cell_qc_distributions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
n_before = len(adata)

# Build boolean masks for each filter independently to report per-filter removal
mask_umi_lo  = adata.obs[col_umi]  >= MIN_UMIS_PER_CELL
mask_umi_hi  = adata.obs[col_umi]  <= upper_umi
mask_gene    = adata.obs[col_gene] >= MIN_GENES_PER_CELL
mask_mito    = adata.obs[col_mito] <= MAX_PCT_MITO

# Report cells failing each individual filter
print('Cells failing each filter (non-exclusive):')
print(f'  UMI < {MIN_UMIS_PER_CELL:,}:      {(~mask_umi_lo).sum():>7,}  ({(~mask_umi_lo).mean():.2%})')
print(f'  UMI > {upper_umi:,}:     {(~mask_umi_hi).sum():>7,}  ({(~mask_umi_hi).mean():.2%})')
print(f'  Genes < {MIN_GENES_PER_CELL}:        {(~mask_gene).sum():>7,}  ({(~mask_gene).mean():.2%})')
print(f'  MT% > {MAX_PCT_MITO}:        {(~mask_mito).sum():>7,}  ({(~mask_mito).mean():.2%})')

# Combined mask: cell must pass ALL filters
cell_mask = mask_umi_lo & mask_umi_hi & mask_gene & mask_mito
n_removed  = n_before - cell_mask.sum()

print(f'\nTotal cells removed (any filter): {n_removed:,}  ({n_removed/n_before:.2%})')
print(f'Cells retained:                   {cell_mask.sum():,}  ({cell_mask.mean():.2%})')

# Apply filter
adata = adata[cell_mask].copy()
print(f'\nPost-cell-QC AnnData: {adata.shape}')

<a id='gene-qc'></a>
## 3. Gene QC — filtering lowly detected genes

### Why filter genes?

A genome-wide expression matrix (e.g., 33,000 genes × 200,000 cells) contains many genes
that are detected in only a handful of cells. These genes:

1. **Contribute noise without signal:** A gene detected in 5 out of 200,000 cells has
   essentially zero statistical power for any downstream test. Its "expression" in those
   5 cells is more likely to represent sequencing noise or ambient RNA than genuine biology.

2. **Inflate memory and compute costs:** Sparse matrix operations are efficient only when
   the sparsity is somewhat uniform. Genes detected in < 0.01% of cells are extreme
   outliers that slow down HVG selection and PCA.

3. **Can distort normalisation:** Very lowly expressed genes with occasional high counts
   (e.g., contaminating transcripts) can artificially inflate library-size estimates.

### Threshold

We require each gene to be detected in at least **`MIN_GENE_CELLS = 10`** cells.
This is a conservative threshold — it removes genes with essentially no detection
without removing genuinely lowly expressed genes that happen to be biologically relevant
in a subset of perturbation groups.

For HVG selection (Step 8), we further restrict to the 2,000 most variable genes.
The gene QC step here is just a coarse pre-filter.

In [ ]:
# Compute the number of cells each gene is detected in
# ncells in adata.var (if pre-computed) or recompute from the count matrix
if 'ncells' in adata.var.columns:
    cells_per_gene = adata.var['ncells'].values
else:
    # Count non-zero entries per gene (column)
    X = adata.X
    cells_per_gene = np.diff(X.tocsc().indptr) if sp.issparse(X) else (X > 0).sum(axis=0).A1

n_genes_before = adata.n_vars
gene_mask      = cells_per_gene >= MIN_GENE_CELLS

print(f'Genes before QC: {n_genes_before:,}')
print(f'Genes detected in < {MIN_GENE_CELLS} cells: {(~gene_mask).sum():,}  ({(~gene_mask).mean():.1%})')
print(f'Genes retained:  {gene_mask.sum():,}')

# Distribution of cells per gene
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(np.log1p(cells_per_gene), bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(np.log1p(MIN_GENE_CELLS), color='red', ls='--', lw=1.5,
                label=f'min={MIN_GENE_CELLS} cells')
axes[0].set_xlabel('log1p(cells per gene)')
axes[0].set_ylabel('Number of genes')
axes[0].set_title('Cells per gene distribution')
axes[0].legend()

axes[1].hist(cells_per_gene[gene_mask], bins=100, color='steelblue', edgecolor='none')
axes[1].set_xlabel('Cells per gene (kept genes)')
axes[1].set_ylabel('Number of genes')
axes[1].set_title(f'Retained genes (>= {MIN_GENE_CELLS} cells)')
plt.tight_layout()
plt.savefig(FIG_DIR / '01a_gene_qc.pdf', bbox_inches='tight')
plt.show()

adata = adata[:, gene_mask].copy()
print(f'\nPost-gene-QC AnnData: {adata.shape}')

<a id='cpp'></a>
## 4. Cells-per-Perturbation Distribution

One of the most informative QC plots in a Perturb-seq experiment is the **distribution of
cell counts per perturbation group**. Unlike bulk RNA-seq where sample size is a design
choice, in a pooled screen it is a result of:

1. **Library balance:** How evenly were different guide RNAs represented in the delivery library?
2. **Sequencing depth:** Were all perturbation groups sequenced to equal depth?
3. **Cell fitness effects:** Did some perturbations kill cells (essential genes) or promote
   proliferation?

### Why this matters for downstream analysis

| Cell count | Statistical capability |
|------------|------------------------|
| < 20       | Cannot perform any reliable DE or E-distance test; exclude |
| 20–200     | Basic Wilcoxon DE is possible; E-distance estimates are noisy |
| 200–1,000  | Reliable E-distance + E-test; good pseudobulk DE |
| > 1,000    | Genome-scale power; can detect very small effect sizes |

### Reading the distribution as biology

The cells-per-perturbation distribution is itself a **biological signal**:

- **Left tail (very few cells):** Perturbations that killed cells or strongly inhibited
  proliferation. In a CRISPRi screen, these are candidate **essential genes** or **negative
  fitness regulators**. A gene that appears in the bottom 1% of cell counts is worth
  following up even before doing DE analysis.

- **Right tail (many more cells than average):** Perturbations that provided a proliferative
  advantage. Knocking down a tumour suppressor or growth inhibitor might produce more cells
  over the screen duration. These are candidate **negative regulators of proliferation**.

- **Bimodal distribution:** Could indicate two delivery batches with different cell numbers,
  or a systematic difference between two guide RNA designs targeting the same gene.

### The three-tier classification

We classify each perturbation into three tiers based on cell count:

- **Tier 1 (pass):** ≥ 200 cells — reliable for E-distance, DE, and all downstream analyses
- **Tier 2 (warn):** 20–199 cells — basic DE possible; flag for caution in E-distance
- **Tier 3 (fail):** < 20 cells — below hard minimum; exclude from all quantitative analyses

In [ ]:
pert_counts = adata.obs['perturbation'].value_counts()
pert_counts_no_ctrl = pert_counts[pert_counts.index != 'control']

print(f'Perturbation groups (excluding control): {len(pert_counts_no_ctrl):,}')
print(f'Total cells in perturbation groups:       {pert_counts_no_ctrl.sum():,}')
print(f'Control cells:                            {pert_counts.get("control", 0):,}')
print()
print('Cell-count summary per perturbation:')
print(pert_counts_no_ctrl.describe().round(1))

# Tier classification
n_tier1 = (pert_counts_no_ctrl >= CELL_PER_PERT_WARN).sum()    # >= 200
n_tier2 = ((pert_counts_no_ctrl >= MIN_CELLS_PER_PERT) & (pert_counts_no_ctrl < CELL_PER_PERT_WARN)).sum()
n_tier3 = (pert_counts_no_ctrl < MIN_CELLS_PER_PERT).sum()     # < 20

print(f'\nTier classification:')
print(f'  Tier 1 (>= {CELL_PER_PERT_WARN} cells, reliable):  {n_tier1:5,}  ({n_tier1/len(pert_counts_no_ctrl):.1%})')
print(f'  Tier 2 ({MIN_CELLS_PER_PERT}-{CELL_PER_PERT_WARN-1} cells, caution):  {n_tier2:5,}  ({n_tier2/len(pert_counts_no_ctrl):.1%})')
print(f'  Tier 3 (<  {MIN_CELLS_PER_PERT} cells, exclude):   {n_tier3:5,}  ({n_tier3/len(pert_counts_no_ctrl):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Linear scale (left) — shows the full distribution shape
axes[0].hist(pert_counts_no_ctrl.values, bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(MIN_CELLS_PER_PERT, color='red',    ls='--', lw=1.5, label=f'hard min ({MIN_CELLS_PER_PERT})')
axes[0].axvline(CELL_PER_PERT_WARN, color='orange', ls='--', lw=1.5, label=f'E-test reliable ({CELL_PER_PERT_WARN})')
ctrl_n = pert_counts.get('control', 0)
axes[0].axvline(ctrl_n, color='green', ls=':', lw=1.5, label=f'control ({ctrl_n:,})')
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Cells per perturbation (linear)')
axes[0].legend(fontsize=8)

# Log scale (right) — better reveals the left tail (essential gene candidates)
axes[1].hist(np.log10(pert_counts_no_ctrl.values + 1), bins=80,
             color='steelblue', edgecolor='none')
axes[1].axvline(np.log10(MIN_CELLS_PER_PERT), color='red',    ls='--', lw=1.5,
                label=f'hard min ({MIN_CELLS_PER_PERT})')
axes[1].axvline(np.log10(CELL_PER_PERT_WARN), color='orange', ls='--', lw=1.5,
                label=f'E-test reliable ({CELL_PER_PERT_WARN})')
axes[1].set_xlabel('log10(cells per perturbation)')
axes[1].set_ylabel('Number of perturbations')
axes[1].set_title('Cells per perturbation (log scale)')
axes[1].legend(fontsize=8)

plt.suptitle('Cells-per-perturbation distribution — ReplogleWeissman2022_rpe1', y=1.02)
plt.savefig(FIG_DIR / '01a_cells_per_perturbation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Assign tier labels to each perturbation group
tier_map = {}
for pert, cnt in pert_counts_no_ctrl.items():
    if cnt >= CELL_PER_PERT_WARN:
        tier_map[pert] = 'tier1_reliable'
    elif cnt >= MIN_CELLS_PER_PERT:
        tier_map[pert] = 'tier2_caution'
    else:
        tier_map[pert] = 'tier3_exclude'
tier_map['control'] = 'control'

adata.obs['pert_cell_tier'] = adata.obs['perturbation'].map(tier_map).fillna('tier3_exclude')

print('pert_cell_tier annotation added to adata.obs.')
print(adata.obs['pert_cell_tier'].value_counts())

# --- Essential gene candidates (Tier 3: very few cells survived) ---
essential_candidates = pert_counts_no_ctrl[pert_counts_no_ctrl < MIN_CELLS_PER_PERT]
print(f'\nPerturbations with < {MIN_CELLS_PER_PERT} cells (essential gene candidates): {len(essential_candidates)}')
if len(essential_candidates) > 0:
    print('  Top 20 (fewest surviving cells):')
    print(essential_candidates.head(20).to_string())

# --- Potential fitness advantage (top cell count, non-control) ---
print(f'\nTop 10 perturbations by cell count (potential fitness advantage):')
print(pert_counts_no_ctrl.head(10).to_string())

# Save the cells-per-perturbation table
cpp_df = pd.DataFrame({
    'perturbation' : pert_counts_no_ctrl.index,
    'n_cells'      : pert_counts_no_ctrl.values,
    'tier'         : [tier_map[p] for p in pert_counts_no_ctrl.index],
})
cpp_df.to_csv(TBL_DIR / '01a_cells_per_perturbation.csv', index=False)
print(f'\nTable saved to {TBL_DIR}/01a_cells_per_perturbation.csv')

<a id='guide-qc'></a>
## 5. Guide Assignment QC — flagging ambiguous assignments

In a Perturb-seq experiment, the link between a cell and its perturbation is established
through guide RNA sequencing. After library preparation, the guide RNA spacer sequence is
read alongside the 10x cell barcode, allowing each cell to be assigned to a perturbation.
This assignment process is not perfect — several failure modes produce cells with uncertain
or incorrect perturbation labels.

### Failure Mode 1: Multiple guides (`nperts > 1`)

The `nperts` field records the number of perturbations assigned to each cell.

- **`nperts == 1`:** Cell received exactly one guide RNA. Standard analysis unit.
- **`nperts == 0`:** Cell received no detectable guide RNA. Could be a failed delivery
  or a cell that was present in the media before guide RNA introduction.
- **`nperts > 1`:** Cell received (or appears to have received) multiple guide RNAs.
  Sources include:
  1. **Cell doublets:** Two cells co-encapsulated in one GEM droplet. The library reflects
     two cells' guide RNA repertoires, making the perturbation label ambiguous.
  2. **Guide RNA library doublets:** The physical library prep contained two guide sequences
     in a single vector (e.g., from lentiviral co-infection at high MOI).
  3. **Sequencing / barcode errors:** Rare, but can assign spurious guide reads to a cell.

**In genome-scale non-combinatorial screens (like Replogle RPE1):** `nperts > 1` cells
are almost always doublets and should be excluded from single-perturbation analyses.

### Failure Mode 2: Low guide coverage (`coverage` / `good_coverage`)

The `coverage` field records the number of guide RNA UMIs detected in each cell.
The `good_coverage` field is a boolean flag set by the scPerturb team:
`True` if coverage ≥ threshold (typically ≥ 5 UMIs), `False` otherwise.

**Why low coverage is problematic:**
- A cell with only 1–3 guide UMIs may have been assigned based on noise (e.g., ambient
  guide RNA from the cell suspension contaminating the droplet).
- The assignment is uncertain: we cannot be confident which gene was actually targeted.
- These cells should be flagged and typically excluded from E-distance analysis.

**Coverage vs. expression:** Note that `coverage` (guide UMI count) is independent of the
total mRNA UMI count. A cell can have excellent mRNA coverage (high ncounts) but poor guide
coverage (ambiguous perturbation identity), or vice versa.

### Failure Mode 3: Guide ID inconsistency

The `guide_id` field contains the actual guide RNA sequence identifier. In well-harmonized
datasets, `guide_id` should map directly to the `perturbation` field. Mismatches can
indicate harmonization errors in the scPerturb pipeline.

### Composite ambiguity flag

We create a composite `guide_ambiguous` flag that is `True` if **any** of the following:
- `nperts != 1` (multi-perturbation or unassigned)
- `good_coverage == False` (if field available)
- `coverage < 5` (if coverage field available)

Ambiguous cells are flagged but **not yet removed** from the AnnData — downstream notebooks
decide how to handle them depending on the analysis.

In [ ]:
print('=== nperts distribution ===')
nperts_counts = adata.obs['nperts'].value_counts().sort_index()
for n, cnt in nperts_counts.items():
    pct = cnt / len(adata) * 100
    bar = '#' * int(pct * 2)
    print(f'  nperts={n}: {cnt:>8,} cells  ({pct:5.1f}%)  {bar}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of nperts distribution
nperts_vals = sorted(adata.obs['nperts'].unique())
nperts_cnts = [nperts_counts.get(n, 0) for n in nperts_vals]
colors_bar  = ['tomato' if n != 1 else 'steelblue' for n in nperts_vals]
axes[0].bar([str(n) for n in nperts_vals], nperts_cnts, color=colors_bar)
axes[0].set_xlabel('nperts (guide RNAs assigned per cell)')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('nperts distribution\n(red = potentially ambiguous)')

# Coverage distribution (if available)
if 'coverage' in adata.obs.columns:
    axes[1].hist(np.log1p(adata.obs['coverage']), bins=80, color='steelblue', edgecolor='none')
    axes[1].axvline(np.log1p(5), color='red', ls='--', lw=1.5, label='coverage < 5 (low)')
    axes[1].set_xlabel('log1p(guide RNA coverage)')
    axes[1].set_ylabel('Cells')
    axes[1].set_title('Guide RNA coverage per cell')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'coverage field not in obs',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title('Guide coverage (not available)')

plt.suptitle('Guide assignment QC — nperts and coverage', y=1.02)
plt.savefig(FIG_DIR / '01a_guide_qc_coverage.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Build composite guide_ambiguous flag ----------------------------------
guide_ambiguous = pd.Series(False, index=adata.obs_names)

# Flag 1: nperts != 1  (multiple guides or no guide)
flag_multi    = adata.obs['nperts'] != 1
guide_ambiguous |= flag_multi
print(f'Cells with nperts != 1:           {flag_multi.sum():>8,}  ({flag_multi.mean():.2%})')

# Flag 2: low guide coverage
if 'good_coverage' in adata.obs.columns:
    flag_low_cov = ~adata.obs['good_coverage'].astype(bool)
    guide_ambiguous |= flag_low_cov
    print(f'Cells with good_coverage=False:   {flag_low_cov.sum():>8,}  ({flag_low_cov.mean():.2%})')
elif 'coverage' in adata.obs.columns:
    COVERAGE_MIN = 5
    flag_low_cov = adata.obs['coverage'] < COVERAGE_MIN
    guide_ambiguous |= flag_low_cov
    print(f'Cells with coverage < {COVERAGE_MIN}:          {flag_low_cov.sum():>8,}  ({flag_low_cov.mean():.2%})')
else:
    print('coverage / good_coverage fields not available — skipping coverage QC.')

# Store flag in adata.obs
adata.obs['guide_ambiguous'] = guide_ambiguous

print(f'\nTotal ambiguous cells (any flag): {guide_ambiguous.sum():>8,}  ({guide_ambiguous.mean():.2%})')
print(f'Clean cells (all flags pass):     {(~guide_ambiguous).sum():>8,}  ({(~guide_ambiguous).mean():.2%})')

# Breakdown by perturbation tier
print('\nAmbiguous flag breakdown by perturbation tier:')
print(adata.obs.groupby('pert_cell_tier')['guide_ambiguous']
      .agg(['sum', 'mean'])
      .rename(columns={'sum': 'n_ambiguous', 'mean': 'frac_ambiguous'})
      .assign(frac_ambiguous=lambda d: d['frac_ambiguous'].map('{:.2%}'.format))
      .to_string())

In [ ]:
# <a id='ctrl-frac'></a>
# === 6. Control fraction check =============================================
# The fraction of non-targeting control cells in the dataset is an important
# QC metric. Too few controls -> poor statistical power for DE analysis.
# Too many controls -> wasted sequencing capacity on non-biological reads.
# Expected range: 10–30% of all cells should be controls.

CONTROL_LABELS = ['control', 'non-targeting', 'safe-harbor', 'CTRL', 'NT']
ctrl_mask  = adata.obs['perturbation'].isin(CONTROL_LABELS)
ctrl_label = adata.obs.loc[ctrl_mask, 'perturbation'].unique()
ctrl_frac  = ctrl_mask.mean()

print('=== Control fraction check ===')
print(f'Control label(s) in this dataset: {ctrl_label.tolist()}')
print(f'Control cells:  {ctrl_mask.sum():,}  ({ctrl_frac:.1%})')
print(f'Perturbed cells: {(~ctrl_mask).sum():,}')
print()
if ctrl_frac < CONTROL_FRAC_MIN:
    print(f'WARNING: Control fraction {ctrl_frac:.1%} is BELOW minimum {CONTROL_FRAC_MIN:.0%}.')
    print('  Impact: E-distance and DE tests may have reduced statistical power.')
    print('  Consider: supplementing with additional non-targeting controls.')
elif ctrl_frac > CONTROL_FRAC_MAX:
    print(f'WARNING: Control fraction {ctrl_frac:.1%} is ABOVE maximum {CONTROL_FRAC_MAX:.0%}.')
    print('  Impact: Inefficient use of sequencing depth (controls are over-represented).')
else:
    print(f'Control fraction {ctrl_frac:.1%} is within expected range [{CONTROL_FRAC_MIN:.0%}, {CONTROL_FRAC_MAX:.0%}]. OK')

<a id='preproc'></a>
## 7. Post-QC Preprocessing

With quality cells and genes retained, we now apply the standard Perturb-seq preprocessing
pipeline to produce representations suitable for downstream analysis:

1. **Library-size normalisation** (`normalize_total`): Scale each cell's counts so that
   the total UMI count equals 10,000. This removes the effect of varying sequencing depth
   across cells. The resulting values are "counts per 10,000" (CP10K).

2. **Log1p transformation** (`log1p`): Apply $\log(1 + x)$ to the normalised counts.
   This compresses the dynamic range (a gene going from 1 to 10 CP10K is biologically
   similar to one going from 100 to 1,000 CP10K, but their raw difference is very different).
   Log-space is also more appropriate for Gaussian assumptions in downstream tests.

3. **Highly variable gene selection** (`highly_variable_genes`, n=2,000): Restrict to
   the 2,000 genes with the highest cell-to-cell variability. These genes carry the most
   biological signal. Housekeeping genes (ribosomal proteins, ubiquitin, etc.) that are
   expressed similarly in all cells contribute noise rather than signal to PCA and UMAP.

4. **Scaling** (`scale`, max_value=10): For each gene, subtract the mean and divide by the
   standard deviation across cells, so that all genes have equal weight in PCA. Clip to
   10 standard deviations to prevent extreme outlier cells from dominating the first PCs.

5. **PCA** (`pca`, n_comps=50): Reduce the 2,000-dimensional HVG space to 50 principal
   components. PCA captures the major axes of variation (e.g., cell cycle, stress response,
   and perturbation effects) while discarding noise. All downstream distance computations
   (E-distance, UMAP, k-NN) use this 50-PC representation.

**Note:** Raw counts are stored in `adata.raw` before normalisation so they can be recovered
for pseudobulk DE (which requires raw counts).

In [ ]:
# Store raw counts BEFORE normalisation for pseudobulk DE (Phase 2 notebook 01c)
adata.raw = adata.copy()
print('Raw counts stored in adata.raw.')

# Step 1: Library-size normalise to 10,000 UMIs per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Step 2: Log1p transform
sc.pp.log1p(adata)

# Step 3: Highly variable gene selection
# Restrict to cells passing guide QC for HVG selection (cleaner HVG estimates)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    subset=False,   # add flag to adata.var; subsetting happens at scale/PCA step
    layer=None,
)
n_hvg = adata.var['highly_variable'].sum()
print(f'\nHVG selection: {n_hvg} genes flagged as highly variable (from {adata.n_vars:,})')

# Step 4: Scale (zero mean, unit variance per gene) and clip outliers
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)

# Fix NaN from zero-variance genes after scaling
X_arr = adata_hvg.X.toarray() if sp.issparse(adata_hvg.X) else adata_hvg.X
if np.isnan(X_arr).any():
    n_nan = np.isnan(X_arr).any(axis=0).sum()
    print(f'Fixing {n_nan} zero-variance genes (NaN -> 0) after scaling.')
    X_arr = np.nan_to_num(X_arr, nan=0.0)
    adata_hvg.X = sp.csr_matrix(X_arr)

# Step 5: PCA
sc.tl.pca(adata_hvg, n_comps=N_PCA, svd_solver='arpack')
var_exp = adata_hvg.uns['pca']['variance_ratio']
print(f'PCA complete.')
print(f'  Variance explained by PC 1–10:  {var_exp[:10].sum():.1%}')
print(f'  Variance explained by all {N_PCA} PCs: {var_exp.sum():.1%}')

# Copy PCA embedding back to main adata
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.uns['pca']    = adata_hvg.uns['pca']
adata.varm['PCs']   = np.zeros((adata.n_vars, N_PCA))  # placeholder for non-HVG genes
adata.varm['PCs'][adata.var['highly_variable']] = adata_hvg.varm['PCs']

# Compute k-NN graph for UMAP (stored in adata — used by 01b)
sc.pp.neighbors(adata, n_neighbors=20, use_rep='X_pca')
print('k-NN graph computed (n_neighbors=20).')

# Scree plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, N_PCA + 1), var_exp * 100, color='steelblue', width=0.8)
ax.axvline(10, color='orange', ls='--', lw=1, label='PC 10')
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot — ReplogleWeissman2022_rpe1 (post-QC, 2000 HVGs)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '01a_scree_plot.pdf', bbox_inches='tight')
plt.show()

print(f'\nFinal processed AnnData: {adata.shape}')

In [ ]:
# <a id='save'></a>
# === 8. Save processed AnnData =============================================
# Save locally to data/processed/scperturb/ and upload to S3.
# The saved AnnData contains:
#   adata.X           — log-normalised expression (10K normalised + log1p)
#   adata.raw         — raw counts (for pseudobulk DE)
#   adata.obs         — all original obs fields + QC flags added here
#   adata.var         — gene metadata + highly_variable flag
#   adata.obsm['X_pca'] — 50-PC embedding
#   adata.obsp['*']   — k-NN graph for UMAP

print(f'Saving to: {PROC_PATH}')
adata.write_h5ad(PROC_PATH, compression='gzip')
print(f'Saved locally ({PROC_PATH.stat().st_size / 1e9:.2f} GB).')

# Upload to S3
s3 = boto3.client('s3')
s3_key = f'processed/scperturb/{PROC_PATH.name}'
try:
    print(f'Uploading to s3://{S3_BUCKET}/{s3_key} ...')
    s3.upload_file(str(PROC_PATH), S3_BUCKET, s3_key)
    print('S3 upload complete.')
except Exception as e:
    print(f'S3 upload failed: {e}')
    print(f'File is available locally at {PROC_PATH}')

# Save a compact QC summary table
qc_summary = pd.DataFrame({
    'perturbation' : adata.obs['perturbation'],
    'n_cells_final': 1,  # each row = 1 cell; group later
}).assign(n_cells_final=1)
qc_per_pert = (
    adata.obs
    .groupby('perturbation')
    .agg(
        n_cells      =('perturbation', 'count'),
        n_ambiguous  =('guide_ambiguous', 'sum'),
        frac_ambiguous=('guide_ambiguous', 'mean'),
        tier         =('pert_cell_tier', 'first'),
    )
    .reset_index()
)
qc_per_pert.to_csv(TBL_DIR / '01a_per_perturbation_qc_summary.csv', index=False)
print(f'Per-perturbation QC summary saved to {TBL_DIR}/01a_per_perturbation_qc_summary.csv')

In [ ]:
# <a id='summary'></a>
print('=' * 65)
print('PHASE 2 QC PREPROCESSING — SUMMARY')
print('=' * 65)
print()
print(f'Dataset: ReplogleWeissman2022_rpe1')
print()

# Compute final stats
n_final = len(adata)
ctrl_n  = (adata.obs['perturbation'] == 'control').sum()
n_perts_t1 = (adata.obs['pert_cell_tier'] == 'tier1_reliable').sum()
n_perts_t2 = (adata.obs['pert_cell_tier'] == 'tier2_caution').sum()
n_ambig  = adata.obs['guide_ambiguous'].sum()

checks = [
    ('File loaded from S3/local',                     True),
    (f'Cells after QC (min {MIN_UMIS_PER_CELL} UMI, min {MIN_GENES_PER_CELL} genes, <{MAX_PCT_MITO}% MT)',
                                                       n_final > 0),
    ('Genes filtered (< 10 cells)',                   adata.n_vars > 0),
    ('Cells-per-perturbation distribution plotted',   True),
    ('Tier 1 (>=200 cells) perturbations',            n_perts_t1 > 0),
    ('Guide ambiguity flag added (guide_ambiguous)',   'guide_ambiguous' in adata.obs.columns),
    ('Control fraction in expected range',             CONTROL_FRAC_MIN <= ctrl_n/n_final <= CONTROL_FRAC_MAX),
    ('Raw counts stored in adata.raw',                adata.raw is not None),
    ('HVG selection complete',                        'highly_variable' in adata.var.columns),
    ('PCA (50 components) computed',                  'X_pca' in adata.obsm),
    ('Processed h5ad saved locally',                  PROC_PATH.exists()),
]

all_pass = True
for check, result in checks:
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print(f'  Final cells:          {n_final:>10,}')
print(f'  Final genes:          {adata.n_vars:>10,}')
print(f'  Control cells:        {ctrl_n:>10,}  ({ctrl_n/n_final:.1%})')
print(f'  Tier 1 perturbations: {(adata.obs["pert_cell_tier"]=="tier1_reliable").nunique():>10,} perturbation groups')
print(f'  Ambiguous guide cells:{n_ambig:>10,}  ({n_ambig/n_final:.1%})')
print()
print('Phase 2 QC checkpoint:', 'PASSED' if all_pass else 'INCOMPLETE (see above)')
print()
print('Next step: 01b_dimred_visualization.ipynb — UMAP + perturbation embeddings')

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'01a_qc_preprocessing_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '01a_qc_preprocessing.ipynb'
)
subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')